# Doctor Identity Resolution – Enterprise (Probabilistic)

Enterprise-grade probabilistic doctor identity resolution using pure Spark SQL.

## Pipeline

1. **Prepare** – Load `silver.doctors`, normalise, reject duplicate keys.
2. **Score**   – Probabilistic scoring: blocking + Fellegi-Sunter weighting.
3. **Cluster** – Connected components on auto-merge + approved edges.
4. **Resolve** – Assign stable `ED-<uuid>` enterprise doctor IDs.
5. **Outputs** – Doctor map, golden doctor dimension, tombstoning.
6. **Audit**   – Validate invariants, write run audit record.


In [ ]:
# No external library installs required.
# All scoring uses native PySpark functions – no Splink dependency.
print("Environment ready.")

In [ ]:
from datetime import datetime, timezone

from config.identity_config import (
    DOCTOR_IDENTITY_TABLES,
    DOCTOR_AUTO_MERGE_THRESHOLD,
    DOCTOR_REVIEW_MIN_THRESHOLD,
)
from src.identity.doctor_preparation import prepare_doctors
from src.identity.doctor_matching    import score_pairs, build_clusters
from src.identity.doctor_resolution  import (
    resolve_enterprise_ids,
    build_analytical_outputs,
    validate_and_audit,
)


## 1. Parameters


In [ ]:
# Parameters (set FULL_REFRESH = True to reprocess all records)
try:
    FULL_REFRESH = dbutils.widgets.get("full_refresh").lower() == "true"
except Exception:
    FULL_REFRESH = False

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
tables = DOCTOR_IDENTITY_TABLES

print(f"run_id       : {RUN_ID}")
print(f"full_refresh : {FULL_REFRESH}")
print(f"source       : {tables['source']}")


## 2. Prepare Doctors


In [ ]:
valid_doctors = prepare_doctors(spark, tables, RUN_ID)


## 3. Probabilistic Scoring


In [ ]:
scored_pairs = score_pairs(spark, valid_doctors)


## 4. Build Clusters


In [ ]:
clusters = build_clusters(
    spark, scored_pairs, valid_doctors, tables,
    DOCTOR_AUTO_MERGE_THRESHOLD, DOCTOR_REVIEW_MIN_THRESHOLD, RUN_ID,
)


## 5. Enterprise ID Resolution


In [ ]:
resolve_enterprise_ids(
    spark, clusters, scored_pairs, valid_doctors, tables, RUN_ID,
)


## 6. Analytical Outputs


In [ ]:
build_analytical_outputs(spark, valid_doctors, tables)


## 7. Validation & Run Audit


In [ ]:
summary = validate_and_audit(
    spark, valid_doctors, tables, RUN_ID, FULL_REFRESH,
)
display(summary)


## 8. Operational Views


In [ ]:
from pyspark.sql import functions as F
from src.identity.writer import table_exists

display(
    spark.table(tables['doctor_map'])
    .groupBy('match_status')
    .agg(
        F.count('*').alias('source_record_count'),
        F.countDistinct('enterprise_doctor_id').alias('enterprise_doctor_count'),
    )
)

display(
    spark.table(tables['registry'])
    .groupBy('enterprise_doctor_id')
    .agg(F.count('*').alias('source_record_count'))
    .filter(F.col('source_record_count') > 1)
    .orderBy(F.desc('source_record_count'))
    .limit(20)
)
